# Single-Cell NicheNet's Ligand Activity Analysis

This notebook shows how NicheNet can predict which ligands might be active in single cells. If a ligand has a high activity in a cell, this means that target genes of that ligand are more strongly expressed in that cell than in other cells.

We use data from Puram et al. (2017) to explore intercellular communication in the tumor microenvironment of HNSCC. We assess the activity of cancer-associated fibroblast (CAF) ligands in malignant cells.

To prioritize ligands regulating a process of interest, we perform a correlation analysis between ligand activities in cells and cell-level scores corresponding to the process (p-EMT in this case).

In [ ]:
import os
os.environ.setdefault("NICHENETR_DATA_DIR", "path/to/nichenetr_data")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nichenetr as nn

### Read in expression data of interacting cells

In [ ]:
hnscc_data = nn.load_hnscc_expression()
expression = hnscc_data["expression"]
sample_info = hnscc_data["sample_info"]

expression_df = pd.DataFrame(
    expression.data.toarray(),
    index=expression.rownames,
    columns=expression.colnames,
)

In [ ]:
# Determine expressed genes in CAFs and malignant cells
tumors_remove = ["HN10", "HN", "HN12", "HN13", "HN24", "HN7", "HN8", "HN23"]

CAF_ids = sample_info[
    (sample_info["Lymph node"] == 0)
    & (~sample_info["tumor"].isin(tumors_remove))
    & (sample_info["non-cancer cell type"] == "CAF")
]["cell"].tolist()

malignant_ids = sample_info[
    (sample_info["Lymph node"] == 0)
    & (~sample_info["tumor"].isin(tumors_remove))
    & (sample_info["classified  as cancer cell"] == 1)
]["cell"].tolist()

def get_expressed_custom(expression_df, cell_ids):
    sub = expression_df.loc[expression_df.index.isin(cell_ids)]
    tpm = 10 * (2 ** sub - 1)
    agg = np.log2(tpm.mean(axis=0) + 1)
    return agg[agg >= 4].index.tolist()

expressed_genes_CAFs = get_expressed_custom(expression_df, CAF_ids)
expressed_genes_malignant = get_expressed_custom(expression_df, malignant_ids)
print(f"Expressed genes in CAFs: {len(expressed_genes_CAFs)}")
print(f"Expressed genes in malignant cells: {len(expressed_genes_malignant)}")

### Load the ligand-target model and define potential ligands

In [ ]:
ligand_target_matrix = nn.load_ligand_target_matrix("human")
lr_network = nn.load_lr_network("human")

ligands = lr_network["from"].unique().tolist()
expressed_ligands = list(set(ligands) & set(expressed_genes_CAFs))

receptors = lr_network["to"].unique().tolist()
expressed_receptors = list(set(receptors) & set(expressed_genes_malignant))

potential_ligands = (
    lr_network[
        lr_network["from"].isin(expressed_ligands)
        & lr_network["to"].isin(expressed_receptors)
    ]["from"].unique().tolist()
)
print(f"Potential ligands: {len(potential_ligands)}")

### Perform single-cell ligand activity analysis

Scale the expression data and run the analysis on 10 example cells from HN5 tumor.

In [ ]:
background_expressed_genes = [
    g for g in expressed_genes_malignant if g in ligand_target_matrix.rownames
]

expression_scaled = nn.scale_quantile(
    expression_df.loc[expression_df.index.isin(malignant_ids), background_expressed_genes].values
)
expression_scaled_df = pd.DataFrame(
    expression_scaled,
    index=[c for c in malignant_ids if c in expression_df.index],
    columns=background_expressed_genes,
)

In [ ]:
# Select 10 cells from HN5 tumor
malignant_hn5_ids = sample_info[
    (sample_info["tumor"] == "HN5")
    & (sample_info["Lymph node"] == 0)
    & (sample_info["classified  as cancer cell"] == 1)
]["cell"].head(10).tolist()

# Filter to cells present in expression_scaled_df
malignant_hn5_ids = [c for c in malignant_hn5_ids if c in expression_scaled_df.index]
print(f"Number of cells for analysis: {len(malignant_hn5_ids)}")

In [ ]:
ligand_activities = nn.predict_single_cell_ligand_activities(
    cell_ids=malignant_hn5_ids,
    expression_scaled=expression_scaled_df,
    ligand_target_matrix=ligand_target_matrix,
    potential_ligands=potential_ligands,
)

ligand_activities.head()

### Ligand prioritization by regression analysis

We score malignant cells on their expression of the core p-EMT gene "TGFBI" and correlate these scores with ligand activities to prioritize p-EMT-inducing ligands.

In [ ]:
# Create cell scores based on TGFBI expression
tgfbi_col = "TGFBI" if "TGFBI" in expression_scaled_df.columns else expression_scaled_df.columns[0]
cell_scores_tbl = pd.DataFrame({
    "cell": malignant_hn5_ids,
    "score": expression_scaled_df.loc[malignant_hn5_ids, tgfbi_col].values,
})
cell_scores_tbl

In [ ]:
# Normalize ligand activities for cross-cell comparison
normalized_ligand_activities = nn.normalize_single_cell_ligand_activities(
    ligand_activities
)
normalized_ligand_activities.head()

In [ ]:
# Correlate ligand activities with cell property scores
output_correlation_analysis = nn.single_ligand_activity_score_regression(
    normalized_ligand_activities, cell_scores_tbl
)

output_correlation_analysis.sort_values("pearson_regression", ascending=False)[
    ["ligand", "pearson_regression"]
].head(20)

### Visualize the relation between ligand activity and cell property score

In [ ]:
# Pick a top ligand to visualize
top_ligand = output_correlation_analysis.sort_values(
    "pearson_regression", ascending=False
)["ligand"].iloc[0]

merged = cell_scores_tbl.merge(normalized_ligand_activities, on="cell")

if top_ligand in merged.columns:
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(merged["score"], merged[top_ligand])
    # Add regression line
    z = np.polyfit(merged["score"], merged[top_ligand], 1)
    p = np.poly1d(z)
    x_line = np.linspace(merged["score"].min(), merged["score"].max(), 100)
    ax.plot(x_line, p(x_line), "r--")
    ax.set_xlabel("Cell score (TGFBI expression)")
    ax.set_ylabel(f"{top_ligand} activity")
    ax.set_title(f"Ligand activity vs. cell score: {top_ligand}")
    plt.tight_layout()
    plt.show()